<a href="https://colab.research.google.com/github/KeithRajbhandari/KRB-HTML/blob/main/Week7_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# Install Streamlit, Pandas, and Plotly
!pip install -q streamlit pandas plotly

# Download and install Cloudflare Tunnel (reliable public URL generator)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.10.0) ...
Setting up cloudflared (2026.10.0) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [20]:
from google.colab import files

print("Please select and upload your ecommerce_sales_analytics_5000.csv file:")
uploaded = files.upload()

Please select and upload your ecommerce_sales_analytics_5000.csv file:


Saving ecommerce_sales_analytics_5000.csv to ecommerce_sales_analytics_5000 (1).csv


In [21]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px

# 1. Page Configuration
st.set_page_config(
    page_title="E-Commerce Sales Analytics Dashboard",
    page_icon="🛍️",
    layout="wide"
)

# 2. Data Loading Directly from Colab Storage
@st.cache_data
def load_data():
    df = pd.read_csv("ecommerce_sales_analytics_5000.csv")
    df['order_date'] = pd.to_datetime(df['order_date'])
    return df

df = load_data()

# 3. Sidebar Filters for Discovery
st.sidebar.title("🛒 Filter Controls")
st.sidebar.markdown("Explore sales performance by filtering through dimensions:")

selected_regions = st.sidebar.multiselect(
    "Select Region(s):",
    options=df['region'].unique(),
    default=df['region'].unique()
)

selected_categories = st.sidebar.multiselect(
    "Select Product Category:",
    options=df['product_category'].unique(),
    default=df['product_category'].unique()
)

min_date = df['order_date'].min().date()
max_date = df['order_date'].max().date()
date_range = st.sidebar.date_input(
    "Order Date Range:",
    value=(min_date, max_date),
    min_value=min_date,
    max_value=max_date
)

filtered_df = df[
    (df['region'].isin(selected_regions)) &
    (df['product_category'].isin(selected_categories)) &
    (df['order_date'].dt.date >= date_range[0]) &
    (df['order_date'].dt.date <= date_range[1])
]

# 4. Main Dashboard Header & KPI Metrics
st.title("🛍️ E-Commerce Sales & Customer Analytics Dashboard")
st.markdown("Interactive information discovery platform built with **Streamlit**, **Pandas**, and **Plotly**.")

if filtered_df.empty:
    st.warning("⚠️ No records match your selected filter criteria. Please adjust your sidebar filters.")
else:
    col1, col2, col3, col4 = st.columns(4)
    total_revenue = filtered_df['revenue'].sum()
    total_orders = len(filtered_df)
    avg_order_val = filtered_df['revenue'].mean() if total_orders > 0 else 0
    avg_rating = filtered_df['customer_rating'].mean() if total_orders > 0 else 0

    col1.metric("Total Revenue", f"${total_revenue:,.2f}")
    col2.metric("Total Orders", f"{total_orders:,}")
    col3.metric("Avg Order Value", f"${avg_order_val:,.2f}")
    col4.metric("Avg Customer Rating", f"{avg_rating:.2f} ⭐")

    st.markdown("---")

    # 5. Interactive Visualizations
    chart_col1, chart_col2 = st.columns(2)

    with chart_col1:
        st.subheader("Revenue by Product Category")
        cat_revenue = filtered_df.groupby('product_category')['revenue'].sum().reset_index()
        fig_bar = px.bar(cat_revenue, x='product_category', y='revenue', color='product_category', title="Total Revenue Across Categories", text_auto='.2s')
        st.plotly_chart(fig_bar, use_container_width=True)

    with chart_col2:
        st.subheader("Sales Trends Over Time")
        daily_sales = filtered_df.groupby(filtered_df['order_date'].dt.date)['revenue'].sum().reset_index()
        fig_line = px.line(daily_sales, x='order_date', y='revenue', title="Daily Revenue Trend", markers=True)
        st.plotly_chart(fig_line, use_container_width=True)

    chart_col3, chart_col4 = st.columns(2)

    with chart_col3:
        st.subheader("Customer Rating vs. Revenue")
        fig_scatter = px.scatter(filtered_df, x='customer_rating', y='revenue', color='region', size='quantity', title="Impact of Customer Ratings on Revenue")
        st.plotly_chart(fig_scatter, use_container_width=True)

    with chart_col4:
        st.subheader("Payment Method Distribution")
        pay_counts = filtered_df['payment_method'].value_counts().reset_index()
        pay_counts.columns = ['payment_method', 'count']
        fig_pie = px.pie(pay_counts, names='payment_method', values='count', title="Preferred Payment Methods", hole=0.4)
        st.plotly_chart(fig_pie, use_container_width=True)

    # 6. Data Table & Export
    st.subheader("📋 Filtered Dataset Records")
    st.dataframe(filtered_df, use_container_width=True)

    csv_export = filtered_df.to_csv(index=False).encode('utf-8')
    st.download_button(
        label="📥 Download Filtered Data as CSV",
        data=csv_export,
        file_name="filtered_ecommerce_sales.csv",
        mime="text/csv"
    )

Overwriting app.py


In [ ]:
# Run Streamlit in the background
!streamlit run app.py &>streamlit.log &

# Launch Cloudflare Tunnel
!cloudflared tunnel --url http://localhost:8501

2026-10-11T15:23:14Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-11T15:23:14Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-11T15:23:18Z INF +--------------------------------------------------------------------------------------------+
2026-10-11T15:23:18Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-11T15:23:18Z INF |  https://distance-zum-pray-expand.trycloudflare.com   